# 11 - E11 Retrieval memory that votes in the logit (ImageNet-C)

E11 tests the post-E10 hypothesis on DeMemte ImageNet: associative memory is
mechanically alive, but soft token mixing is too weak to change predictions.
Here the memory contributes a cache/kNN score directly at the classifier decision:

`logits_final = logits_base + alpha_eff(x) * logits_cache`

Base: `dememte_imagenet_resnet50_vqsa`.  The primary key is `zq_pool`; `z_pool`
and `fused` are ablations. Source caches are built from clean ImageNet train
examples; evaluation uses clean ImageNet val plus real ImageNet-C conditions.

Oracle/eval-label caches are diagnostic only and disabled by default for the full
ImageNet-C run.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'dememte').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)


repo root: /home/nakato/projects/Dememte


In [2]:
import json

import numpy as np
import pandas as pd
import torch

from dememte.config import E6Config
from dememte.data import build_imagenet_c_loader, build_imagenet_loaders, seed_everything
from dememte.evaluation import RETRIEVAL_DIAG_KEYS, VQSA_KEYS, evaluate_dememte, evaluate_dememte_tta
from dememte.io import ensure_dir, load_checkpoint, write_csv, write_json
from dememte.models import make_dememte_variant
from dememte.retrieval import RetrievalConfig, RetrievalLogitAdapter, build_labeled_cache

BASE = 'dememte_imagenet_resnet50_vqsa'
OUT = ensure_dir(ROOT / 'notebooks' / '11_retrieval_memory' / 'out')

DATA_ROOT_CLEAN = ROOT / 'experiments' / 'data' / 'imagenet-clean-5k'
DATA_ROOT_C = ROOT / 'experiments' / 'data' / 'imagenet-c-subset'
IMAGENET_OUT = ROOT / 'experiments' / 'imagenet_dememte' / 'out'
DEMEMTE_CHECKPOINT = IMAGENET_OUT / 'dememte_imagenet_resnet50_vqsa_best.pt'
TRAIN_CONFIG_PATH = IMAGENET_OUT / 'train_config.json'

CORRUPTIONS = ['gaussian_noise', 'motion_blur', 'pixelate', 'jpeg_compression']
SEVERITIES = [3, 5]
MAX_SAMPLES_PER_CLASS = None
BATCH_SIZE = 64
NUM_WORKERS = 4
MAX_CACHE_ITEMS = None
RUN_ORACLE_DIAGNOSTIC = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
print('base:', BASE)
print('imagenet clean:', DATA_ROOT_CLEAN)
print('imagenet-c:', DATA_ROOT_C)


device: cuda
base: dememte_imagenet_resnet50_vqsa
imagenet clean: /home/nakato/projects/Dememte/experiments/data/imagenet-clean-5k
imagenet-c: /home/nakato/projects/Dememte/experiments/data/imagenet-c-subset


## Data and checkpoint


In [3]:
def load_imagenet_cfg():
    if TRAIN_CONFIG_PATH.exists():
        payload = json.loads(TRAIN_CONFIG_PATH.read_text(encoding='utf-8'))
        cfg = E6Config(**payload['config'])
    else:
        cfg = E6Config(
            dataset='imagenet',
            data_dir=str(DATA_ROOT_CLEAN),
            num_classes=1000,
            backbone_name='resnet50',
            backbone_out_channels=2048,
            quantizer_type='ema_vq',
            vq_kmeans_init=False,
            dead_code_restart=False,
        )
    cfg.dataset = 'imagenet_c'
    cfg.data_dir = str(DATA_ROOT_C)
    cfg.num_classes = 1000
    cfg.batch_size = BATCH_SIZE
    cfg.num_workers = NUM_WORKERS
    cfg.device = device
    return cfg


cfg = load_imagenet_cfg()
seed_everything(cfg.seed)

clean_train_loader, clean_val_loader, clean_meta = build_imagenet_loaders(
    DATA_ROOT_CLEAN,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    seed=cfg.seed,
)
meta = {
    'dataset': 'imagenet_c',
    'clean_meta': clean_meta,
    'imagenet_c_root': str(DATA_ROOT_C),
    'corruptions': CORRUPTIONS,
    'severities': SEVERITIES,
    'max_samples_per_class': MAX_SAMPLES_PER_CLASS,
    'max_cache_items': MAX_CACHE_ITEMS,
}
print(meta)

ckpt = DEMEMTE_CHECKPOINT
if not ckpt.exists():
    raise FileNotFoundError(
        f'Missing DeMemte ImageNet checkpoint: {ckpt}. '
        'Run notebooks/06b_imagenet_train/e6_imagenet_train.ipynb first.'
    )


def load_base_model():
    model = make_dememte_variant(cfg, device=device)
    load_checkpoint(model, ckpt, device=device, strict=True)
    model.eval()
    model.requires_grad_(False)
    return model


def imagenet_c_loader(corruption, severity, *, max_samples_per_class=MAX_SAMPLES_PER_CLASS):
    return build_imagenet_c_loader(
        DATA_ROOT_C,
        corruption,
        severity,
        batch_size=cfg.batch_size,
        num_workers=cfg.num_workers,
        max_samples_per_class=max_samples_per_class,
        seed=cfg.seed,
    )


{'dataset': 'imagenet_c', 'clean_meta': {'dataset': 'imagenet', 'root': '/home/nakato/projects/Dememte/experiments/data/imagenet-clean-5k', 'train_size': 5000, 'val_size': 1000, 'num_classes': 1000}, 'imagenet_c_root': '/home/nakato/projects/Dememte/experiments/data/imagenet-c-subset', 'corruptions': ['gaussian_noise', 'motion_blur', 'pixelate', 'jpeg_compression'], 'severities': [3, 5], 'max_samples_per_class': None, 'max_cache_items': None}


## Build source and oracle caches


In [4]:
source_model_for_cache = load_base_model()
source_caches = {
    'zq_pool': build_labeled_cache(
        source_model_for_cache,
        clean_train_loader,
        device=device,
        key_space='zq_pool',
        num_classes=cfg.num_classes,
        max_items=MAX_CACHE_ITEMS,
    ),
    'z_pool': build_labeled_cache(
        source_model_for_cache,
        clean_train_loader,
        device=device,
        key_space='z_pool',
        num_classes=cfg.num_classes,
        max_items=MAX_CACHE_ITEMS,
    ),
    'fused': build_labeled_cache(
        source_model_for_cache,
        clean_train_loader,
        device=device,
        key_space='fused',
        num_classes=cfg.num_classes,
        max_items=MAX_CACHE_ITEMS,
    ),
}
{k: v.size for k, v in source_caches.items()}


{'zq_pool': 5000, 'z_pool': 5000, 'fused': 5000}

## Variants


In [5]:
variants = [
    ('source_cache_zq_pool_fixed_alpha',
     RetrievalConfig(key_space='zq_pool', alpha_mode='fixed', alpha_max=1.0, cache_source=True, episodic_size=0),
     source_caches['zq_pool']),
    ('source_cache_z_pool_fixed_alpha',
     RetrievalConfig(key_space='z_pool', alpha_mode='fixed', alpha_max=1.0, cache_source=True, episodic_size=0),
     source_caches['z_pool']),
    ('source_cache_fused_fixed_alpha',
     RetrievalConfig(key_space='fused', alpha_mode='fixed', alpha_max=1.0, cache_source=True, episodic_size=0),
     source_caches['fused']),
    ('source_cache_zq_pool_unfamiliarity_alpha',
     RetrievalConfig(key_space='zq_pool', alpha_mode='unfamiliarity', alpha_max=1.0, cache_source=True, episodic_size=0),
     source_caches['zq_pool']),
    ('episodic_cache_zq_pool',
     RetrievalConfig(key_space='zq_pool', alpha_mode='unfamiliarity', alpha_max=1.0, cache_source=False),
     None),
    ('dual_cache_zq_pool',
     RetrievalConfig(key_space='zq_pool', alpha_mode='unfamiliarity', alpha_max=1.0, cache_source=True),
     source_caches['zq_pool']),
]
[(name, cfg_variant.key_space, cfg_variant.alpha_mode) for name, cfg_variant, _ in variants]


[('source_cache_zq_pool_fixed_alpha', 'zq_pool', 'fixed'),
 ('source_cache_z_pool_fixed_alpha', 'z_pool', 'fixed'),
 ('source_cache_fused_fixed_alpha', 'fused', 'fixed'),
 ('source_cache_zq_pool_unfamiliarity_alpha', 'zq_pool', 'unfamiliarity'),
 ('episodic_cache_zq_pool', 'zq_pool', 'unfamiliarity'),
 ('dual_cache_zq_pool', 'zq_pool', 'unfamiliarity')]

## Run E11


In [6]:
def scalar_values(record):
    return {k: v for k, v in record.items() if isinstance(v, (int, float, bool, str, np.floating))}


def summarize_records(clean_record, corruption_records):
    acc_by_corr = {
        f'corrupt_acc_{corr}': float(np.mean([r['acc'] for r in records]))
        for corr, records in corruption_records.items()
    }
    all_corrupt = [r for records in corruption_records.values() for r in records]
    metrics = {
        'clean_acc': clean_record['acc'],
        'corrupt_acc_avg': float(np.mean(list(acc_by_corr.values()))) if acc_by_corr else 0.0,
        **acc_by_corr,
        'ece_clean': clean_record.get('ece', 0.0),
        'ece_corrupt_avg': float(np.mean([r.get('ece', 0.0) for r in all_corrupt])) if all_corrupt else 0.0,
        'nll_clean': clean_record.get('nll', 0.0),
        'nll_corrupt_avg': float(np.mean([r.get('nll', 0.0) for r in all_corrupt])) if all_corrupt else 0.0,
        'brier_clean': clean_record.get('brier', 0.0),
        'brier_corrupt_avg': float(np.mean([r.get('brier', 0.0) for r in all_corrupt])) if all_corrupt else 0.0,
        'corruption_records': corruption_records,
        'clean_record': clean_record,
    }
    for key in VQSA_KEYS:
        mean_key = f'{key}_mean'
        if mean_key in clean_record:
            metrics[f'{key}_clean'] = clean_record[mean_key]
            metrics[f'{key}_corrupt_avg'] = float(np.mean([r.get(mean_key, 0.0) for r in all_corrupt])) if all_corrupt else 0.0
    for key in RETRIEVAL_DIAG_KEYS:
        mean_key = f'{key}_mean'
        if mean_key in clean_record:
            metrics[f'{key}_clean'] = clean_record[mean_key]
            metrics[f'{key}_corrupt_avg'] = float(np.mean([r.get(mean_key, 0.0) for r in all_corrupt])) if all_corrupt else 0.0
    return metrics


def curve_rows_for(variant_name, label, clean_record, corruption_records):
    rows = []
    clean_row = {'variant': variant_name, 'model': label, 'corruption': 'clean', 'severity': 0}
    clean_row.update(scalar_values(clean_record))
    rows.append(clean_row)
    for corr, records in corruption_records.items():
        for severity, record in zip(SEVERITIES, records):
            row = {'variant': variant_name, 'model': label, 'corruption': corr, 'severity': severity}
            row.update(scalar_values(record))
            rows.append(row)
    return rows


def write_markdown_table(rows, path):
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    df = pd.DataFrame(rows)
    cols = [
        'variant', 'clean_acc', 'corrupt_acc_avg', 'delta_corrupt_vs_source',
        'ece_corrupt_avg', 'nll_corrupt_avg', 'flip_rate_corrupt_avg',
        'corrected_by_retrieval_corrupt_avg', 'broken_by_retrieval_corrupt_avg',
        'retrieval_alpha_corrupt_avg', 'diagnostic_only',
    ]
    cols = [c for c in cols if c in df.columns]
    path.write_text(df[cols].to_markdown(index=False), encoding='utf-8')


def evaluate_source_records(model):
    clean_record = evaluate_dememte(model, clean_val_loader, device=device)
    corruption_records = {}
    for corruption in CORRUPTIONS:
        corruption_records[corruption] = []
        for severity in SEVERITIES:
            loader, cmeta = imagenet_c_loader(corruption, severity)
            print(f'  source :: {corruption}/{severity} ({cmeta["size"]} samples)')
            corruption_records[corruption].append(evaluate_dememte(model, loader, device=device))
    return clean_record, corruption_records


def evaluate_adapter_records(factory, variant_name):
    clean_record = evaluate_dememte_tta(
        factory(),
        clean_val_loader,
        device=device,
        tta_method=variant_name,
        tta_base_variant=BASE,
    )
    corruption_records = {}
    for corruption in CORRUPTIONS:
        corruption_records[corruption] = []
        for severity in SEVERITIES:
            loader, cmeta = imagenet_c_loader(corruption, severity)
            print(f'     {corruption}/{severity} ({cmeta["size"]} samples)')
            corruption_records[corruption].append(evaluate_dememte_tta(
                factory(),
                loader,
                device=device,
                tta_method=variant_name,
                tta_base_variant=BASE,
            ))
    return clean_record, corruption_records


all_summaries = []
all_curves = []

src_model = load_base_model()
print('=== SOURCE:', BASE, '===')
src_clean, src_corrupt = evaluate_source_records(src_model)
src_metrics = summarize_records(src_clean, src_corrupt)
src_summary = {k: v for k, v in src_metrics.items() if isinstance(v, (int, float, bool, str, np.floating))}
src_summary.update({
    'variant': 'source',
    'label': f'{BASE}::source',
    'base_variant': BASE,
    'base_checkpoint': str(ckpt),
    'dataset': meta['dataset'],
    'clean_root': str(DATA_ROOT_CLEAN),
    'imagenet_c_root': str(DATA_ROOT_C),
    'corruptions': ','.join(CORRUPTIONS),
    'severities': ','.join(str(s) for s in SEVERITIES),
    'max_samples_per_class': MAX_SAMPLES_PER_CLASS,
    'max_cache_items': MAX_CACHE_ITEMS,
    'quantizer_type': cfg.quantizer_type,
    'delta_clean_vs_source': 0.0,
    'delta_corrupt_vs_source': 0.0,
    'diagnostic_only': False,
})
all_summaries.append(src_summary)
all_curves.extend(curve_rows_for('source', src_summary['label'], src_clean, src_corrupt))

for variant_name, r_cfg, cache in variants:
    print('--', variant_name)

    def factory(r_cfg=r_cfg, cache=cache):
        model = load_base_model()
        return RetrievalLogitAdapter(model, r_cfg, source_cache=cache, num_classes=cfg.num_classes)

    clean_record, corrupt_records = evaluate_adapter_records(factory, variant_name)
    metrics = summarize_records(clean_record, corrupt_records)
    label = f'{BASE}::{variant_name}'
    curve_rows = curve_rows_for(variant_name, label, clean_record, corrupt_records)
    summary = {k: v for k, v in metrics.items() if isinstance(v, (int, float, bool, str, np.floating))}
    summary.update({
        'variant': variant_name,
        'label': label,
        'base_variant': BASE,
        'base_checkpoint': str(ckpt),
        'dataset': meta['dataset'],
        'clean_root': str(DATA_ROOT_CLEAN),
        'imagenet_c_root': str(DATA_ROOT_C),
        'corruptions': ','.join(CORRUPTIONS),
        'severities': ','.join(str(s) for s in SEVERITIES),
        'max_samples_per_class': MAX_SAMPLES_PER_CLASS,
        'max_cache_items': MAX_CACHE_ITEMS,
        'quantizer_type': cfg.quantizer_type,
        'delta_clean_vs_source': metrics['clean_acc'] - src_summary['clean_acc'],
        'delta_corrupt_vs_source': metrics['corrupt_acc_avg'] - src_summary['corrupt_acc_avg'],
        'diagnostic_only': False,
    })
    all_summaries.append(summary)
    all_curves.extend(curve_rows)

    method_dir = ensure_dir(OUT / variant_name)
    write_json(summary, method_dir / 'metrics.json')
    write_csv(curve_rows, method_dir / 'signal_curves.csv')

    print({
        k: round(float(summary[k]), 4)
        for k in ['clean_acc', 'corrupt_acc_avg', 'delta_corrupt_vs_source',
                  'flip_rate_corrupt_avg', 'corrected_by_retrieval_corrupt_avg',
                  'broken_by_retrieval_corrupt_avg', 'retrieval_alpha_corrupt_avg']
        if k in summary
    })

if RUN_ORACLE_DIAGNOSTIC:
    variant_name = 'oracle_eval_cache_zq_pool_diagnostic_only'
    print('--', variant_name)
    oracle_clean_cache = build_labeled_cache(
        load_base_model(), clean_val_loader, device=device, key_space='zq_pool', num_classes=cfg.num_classes
    )
    oracle_cfg = RetrievalConfig(key_space='zq_pool', alpha_mode='fixed', alpha_max=1.0, cache_source=True, episodic_size=0)
    clean_record = evaluate_dememte_tta(
        RetrievalLogitAdapter(load_base_model(), oracle_cfg, source_cache=oracle_clean_cache, num_classes=cfg.num_classes),
        clean_val_loader,
        device=device,
        tta_method=variant_name,
        tta_base_variant=BASE,
    )
    corrupt_records = {}
    for corruption in CORRUPTIONS:
        corrupt_records[corruption] = []
        for severity in SEVERITIES:
            loader, cmeta = imagenet_c_loader(corruption, severity)
            print(f'     oracle {corruption}/{severity} ({cmeta["size"]} samples)')
            oracle_cache = build_labeled_cache(
                load_base_model(), loader, device=device, key_space='zq_pool', num_classes=cfg.num_classes
            )
            adapter = RetrievalLogitAdapter(
                load_base_model(), oracle_cfg, source_cache=oracle_cache, num_classes=cfg.num_classes
            )
            corrupt_records[corruption].append(evaluate_dememte_tta(
                adapter,
                loader,
                device=device,
                tta_method=variant_name,
                tta_base_variant=BASE,
            ))
    metrics = summarize_records(clean_record, corrupt_records)
    label = f'{BASE}::{variant_name}'
    summary = {k: v for k, v in metrics.items() if isinstance(v, (int, float, bool, str, np.floating))}
    summary.update({
        'variant': variant_name,
        'label': label,
        'base_variant': BASE,
        'base_checkpoint': str(ckpt),
        'dataset': meta['dataset'],
        'delta_clean_vs_source': metrics['clean_acc'] - src_summary['clean_acc'],
        'delta_corrupt_vs_source': metrics['corrupt_acc_avg'] - src_summary['corrupt_acc_avg'],
        'diagnostic_only': True,
    })
    all_summaries.append(summary)
    all_curves.extend(curve_rows_for(variant_name, label, clean_record, corrupt_records))

ranked = sorted(all_summaries, key=lambda r: r.get('corrupt_acc_avg', 0.0), reverse=True)
write_csv(ranked, OUT / 'e11_results.csv')
write_csv(all_curves, OUT / 'e11_curves.csv')
write_markdown_table(ranked, OUT / 'e11_summary.md')
pd.DataFrame(ranked).head(20)


=== SOURCE: dememte_imagenet_resnet50_vqsa ===


  source :: gaussian_noise/3 (50000 samples)


  source :: gaussian_noise/5 (50000 samples)


  source :: motion_blur/3 (50000 samples)


  source :: motion_blur/5 (50000 samples)


...

     pixelate/5 (50000 samples)


     jpeg_compression/3 (50000 samples)


     jpeg_compression/5 (50000 samples)


{'clean_acc': 0.617, 'corrupt_acc_avg': 0.2073, 'delta_corrupt_vs_source': -0.0101, 'flip_rate_corrupt_avg': 0.3852, 'corrected_by_retrieval_corrupt_avg': 0.0193, 'broken_by_retrieval_corrupt_avg': 0.0294, 'retrieval_alpha_corrupt_avg': 0.6358}


,clean_acc,corrupt_acc_avg,corrupt_acc_gaussian_noise,corrupt_acc_motion_blur,corrupt_acc_pixelate,corrupt_acc_jpeg_compression,ece_clean,ece_corrupt_avg,nll_clean,nll_corrupt_avg,...,clean_root,imagenet_c_root,corruptions,severities,max_samples_per_class,max_cache_items,quantizer_type,delta_clean_vs_source,delta_corrupt_vs_source,diagnostic_only
0,0.636,0.242362,0.16471,0.15258,0.26885,0.38331,0.172294,0.188611,2.078460,5.124382,...,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,None,ema_vq,0.022,0.024960,False
1,0.630,0.225400,0.15088,0.13870,0.24982,0.36220,0.159728,0.177090,2.112308,5.165474,...,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,None,ema_vq,0.016,0.007997,False
2,0.618,0.219995,0.14409,0.13607,0.24498,0.35484,0.156538,0.186101,2.096007,5.168384,...,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,None,ema_vq,0.004,0.002592,False
3,0.614,0.217402,0.14533,0.13268,0.24084,0.35076,0.131667,0.146770,2.048999,5.110213,...,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,None,ema_vq,0.000,0.000000,False
4,0.612,0.216537,0.14075,0.13498,0.24108,0.34934,0.179176,0.208034,2.279401,5.274599,...,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,None,ema_vq,-0.002,-0.000865,False
5,0.617,0.207338,0.13333,0.12445,0.23129,0.34028,0.158186,0.298499,2.119227,5.771445,...,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,None,ema_vq,0.003,-0.010065,False
6,0.611,0.203852,0.13172,0.12196,0.22594,0.33579,0.135816,0.273919,2.071149,5.707124,...,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,None,ema_vq,-0.003,-0.013550,False


## Interpretation guardrails

- Source caches are built from clean ImageNet train labels; they are valid for the
  standard ImageNet-C run.
- `RUN_ORACLE_DIAGNOSTIC=True` builds eval-label caches per condition and is only
  a headroom diagnostic. Do not report it as a valid test-time method.
- A positive E11 result must improve or repair flips with bounded ECE/NLL; cache
  activity alone is not enough.
- Before claiming a stable gain, rerun with multiple batch-order seeds.
